# Ordered Logistic Regression Results for Adoption Predictors (FAIR²) Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. The dataset contains ordered logistic regression outputs and associated metadata for knowledge adoption in rangeland management practices in Northern Kenya.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced by their `@id` fields, as required for FAIR datasets and Croissant schemas.

In [ ]:
# List all record sets by their @id and provide an overview of their fields

print("Available record sets (referenced by @id):")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | Name: {rs.get('name', '(no name)')}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields:")
        for field in fields:
            try:
                fid = field['@id']
                fname = field.get('name', '')
                print(f"    - Field @id: {fid} | Name: {fname}")
            except Exception:
                print(f"    - Field: {field}")
    print()

# For illustration, print a few records from each record set
for rs in record_sets:
    print(f"Sample records from RecordSet @id: {rs['@id']}")
    try:
        records = dataset.records(record_set=rs['@id'])
        for i, rec in enumerate(records):
            print(f"  Record {i+1}: {rec}")
            if i == 2:
                break
    except Exception as e:
        print(f"  Could not load records: {e}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s listed in the previous overview.

In [ ]:
# Extract data from each record set into a dictionary of DataFrames

dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records from RecordSet @id: {rs_id} (columns: {df.columns.tolist()})")
        else:
            print(f"No records loaded for RecordSet @id: {rs_id}")
    except Exception as e:
        print(f"Could not load records for RecordSet @id: {rs_id}: {e}")

# Choose a record set with a non-empty DataFrame for analysis
if len(dataframes) == 0:
    print("No record sets with data were found!")
else:
    # Select the first available record set for demonstration
    analysis_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in selected RecordSet (@id: {analysis_record_set_id}):")
    print(dataframes[analysis_record_set_id].columns.tolist())
    dataframes[analysis_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic filtering, normalization, and grouping operations on a numeric field. All field references use their `@id`s to ensure clear provenance.

In [ ]:
# Ensure there is at least one DataFrame to work with
if len(dataframes) == 0:
    print("No data available for EDA.")
else:
    df = dataframes[analysis_record_set_id]
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_candidates:
        # Fall back to try to convert a known field
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                pass
        numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()

    if not numeric_candidates:
        print("No numeric fields available for EDA.")
    else:
        # Select the first available numeric field for demonstration
        numeric_field_id = numeric_candidates[0]
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 1
        print(f"Analyzing numeric field by @id: {numeric_field_id}, using threshold {threshold}")
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 0
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by a non-numeric field (if available)
        group_field_candidates = [c for c in df.columns if c != numeric_field_id and df[c].nunique() > 1]
        group_field_id = group_field_candidates[0] if group_field_candidates else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field for grouping analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) == 0 or not numeric_candidates:
    print("No data available for visualization.")
else:
    # Histogram for the selected numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    # If grouping field exists, visualize group means
    if group_field_id:
        plt.figure(figsize=(8, 4))
        grouped_df.plot(kind='bar')
        plt.title(f'Average {numeric_field_id} by {group_field_id}')
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² dataset using the mlcroissant library, referencing all entities by their Croissant `@id`. You can extend this notebook with deeper domain-specific analyses and visualizations based on your research or policy needs.